#### 基于Qwen2.5-0.5B 基础预训练模型和LoRA微调的的用户交易意图分析分类微调模型

##### 加载Qwen2.5-0.5B模型
 - 使用ModelScope对Transformers包的封装加载Transformers的AutoModel和AutoTokenizer;
 - 分别加载模型和分词器
##### 调整模型输出层
- 添加一个分类层
- 维度为896、in_features=896、out_features=2

In [80]:
from modelscope import AutoModelForSequenceClassification, AutoTokenizer

model_path = "Qwen/Qwen2.5-0.5B"

model = (AutoModelForSequenceClassification.from_pretrained(pretrained_model_name_or_path=model_path, num_labels=2)).to("cuda")
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_path)

2026-03-10 15:27:38,787 - modelscope - INFO - Target directory already exists, skipping creation.
Some weights of Qwen2ForSequenceClassification were not initialized from the model checkpoint at /root/.cache/modelscope/hub/models/Qwen/Qwen2___5-0___5B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


2026-03-10 15:27:45,076 - modelscope - INFO - Target directory already exists, skipping creation.


##### 模型测试
- 使用cuda0设备、token IDS、res_token_ids转PyTorch Tensor

In [94]:
token_ids = tokenizer(["这个活动的后续流程是怎样的？"], return_tensors="pt", padding=True, truncation=True).to("cuda")
token_ids.input_ids[-1, :]
res_token_ids = model(**token_ids)
res_token_ids

SequenceClassifierOutputWithPast(loss=None, logits=tensor([[-0.2898, -2.6886]], device='cuda:0', grad_fn=<IndexBackward0>), past_key_values=DynamicCache(layers=[DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer]), hidden_states=None, attentions=None)

##### 初始化LoRA微调
预计微调500万参数
- 注意安装PEFT包（Powered By Transformers）；
- 初始化LoraConfig，包含任务类型（因果语言模型，即Transformers）；
- 设置rank以及缩放、设置暂退率为0.1同时配置需要融合LoRA层的原始模型层（应用到全部线性层）
- 创建LoRA Layer、LoRA With Linear、replace_linear_with_lora

In [92]:
from peft import LoraConfig, get_peft_model, TaskType

lora_conf = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "score"],
    bias="none"
)

trading_intent_classification_model = get_peft_model(model=model, peft_config=lora_conf)
trading_intent_classification_model.print_trainable_parameters()

/miniconda/envs/d2l/lib/python3.9/site-packages/peft/mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/miniconda/envs/d2l/lib/python3.9/site-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 8,800,000 || all params: 502,834,560 || trainable%: 1.7501


##### 加载数据集
- 分批加载训练集、验证集、测试集
- 数据预处理：对数据进行padding或truncation并将数据文本部分分别进行token化

In [91]:
from datasets import load_dataset

dataset = load_dataset("csv", data_files={"train": "/deps/athena/classification_lora_train.csv", "val": "/deps/athena/classification_lora_val.csv", "test": "/deps/athena/classification_lora_test.csv"})

tokenized_datasets = dataset.map(
    function=lambda examples: tokenizer(examples['Text'], truncation=True, max_length=512, padding="max_length"),
    batched=True,
    remove_columns=['Text']
)